In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder

# Load local CSV
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

print(f"Loaded {len(df)} rows")
print(f"Columns: {df.columns.tolist()}")

# Clean data - remove rows with missing values
df = df.dropna()
print(f"After cleaning: {len(df)} rows")

# Create features from your data
# Using columns that exist in your data
df['topic_encoded'] = LabelEncoder().fit_transform(df['content_hash_id'].astype(str))  # Using content_hash_id as proxy
df['author_encoded'] = LabelEncoder().fit_transform(df['client_hash_id'].astype(str))  # Using client_hash_id as proxy

# Select features and target
# TARGET: What are you predicting? You don't have 'views_7d' visible
# Let's use 'gsc_clicks' or 'gsc_impressions' as proxy
features = ['gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'scroll_events', 'month']
X = df[features]

# TARGET: Using gsc_impressions as a proxy for views
# If you have 'views_7d', use that instead
y = df['gsc_impressions']  # Or 'gsc_clicks'

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Train: {len(X_train)} rows")
print(f"Test: {len(X_test)} rows")

# Train Random Forest
model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

# Predictions
train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

# Calculate metrics
train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))
test_mae = mean_absolute_error(y_test, test_pred)
test_r2 = r2_score(y_test, test_pred)

print("\n" + "=" * 60)
print("MODEL PERFORMANCE")
print("=" * 60)
print(f"Train RMSE: {train_rmse:.2f}")
print(f"Test RMSE: {test_rmse:.2f}")
print(f"Test MAE: {test_mae:.2f}")
print(f"Test R²: {test_r2:.3f}")

# BASELINE COMPARISON
print("\n" + "=" * 60)
print("BASELINE COMPARISON")
print("=" * 60)

# Your baseline from Week 4
baseline_rmse = 1200
improvement = ((baseline_rmse - test_rmse) / baseline_rmse) * 100

print(f"Baseline RMSE: {baseline_rmse:.2f}")
print(f"Model RMSE: {test_rmse:.2f}")
print(f"Improvement: {improvement:.1f}%")

if test_rmse < baseline_rmse:
    print("✅ Model BEATS the baseline!")
else:
    print("⚠️ Model does NOT beat the baseline. Try tuning.")

# Feature Importance
print("\n" + "=" * 60)
print("FEATURE IMPORTANCE")
print("=" * 60)
importance = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print(importance)

FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/content_refresh_anonymized.csv'

dim_clients                     104 rows
dim_content                 519,606 rows
fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


My lane: Content Performance Forecasting
Why: I want to predict article views to help content teams prioritize.


Search question: What will be the 7-day view count for this article?
Decision: Which draft to work on next
Action: Prioritize top 3 predicted articles
Cost of wrong call: Wasting time on low-performing content
